In [9]:
from IPython.display import IFrame

IFrame('https://www.youtube.com/embed/38aMTXY2usU', width=560, height=315)

Video Link : https://youtu.be/38aMTXY2usU?si=biEsG2w4QLOBFyt7

In [10]:
! pip install -q --upgrade langchain langchain-openai langchain-core langchain_community docx2txt pypdf  langchain_chroma sentence_transformers

### **Part 1:**

- **Understanding RAG (Retrieval-Augmented Generation)**
    - What is RAG?
    - Importance of RAG in AI applications
- **Introduction to LangChain and LCEL**
    - Overview of LangChain
    - LangChain Expression Language (LCEL)
- **Exploring LangChain Components**
    - LLM (Large Language Models)
    - Prompts
    - Retrievers
    - Composing components into a chain
- **Document Processing and Vector Databases**
    - Document splitting techniques
    - Embedding documents
    - Storing and retrieving documents in a vector database

- **Creating Your First RAG Chain**
    - Step-by-step guide to building a basic RAG chain
    - Answering questions from documents using RAG
- **Building a Conversational RAG**
    - Handling follow-up questions
    - Concept of contextualizing and refining queries
- **Using pre-built LangChain RAG chains**
    - history_aware_retriever
    - create_retrieval_chain
- **Building Multi User Chatbot**
    - Managing conversation history using a database table

### **Part 2: Moving to Production with FastAPI**

- **Integrating Colab Code with FastAPI**
    - Setting up FastAPI for production
    - Modularizing code into different files for maintainability
    - Implemeting chatbot endpoint to talk to your data
- **Creating API Endpoints**
    - Building endpoints for file upload, list, and deletion
    - Testing and validating endpoints

- **Building a Streamlit Interface**
    - Creating a user-friendly interface with Streamlit
    - Integrating the RAG chatbot API with the Streamlit app
    - File management features (upload, list, delete)

###What is Retrieval Augmented Generation (RAG)?
RAG is a technique that enhances language models by combining them with a retrieval system. It allows the model to access and utilize external knowledge when generating responses.

The process typically involves:
####Indexing a large corpus of documents

In [11]:
import langchain
print(langchain.__version__)

1.3.14


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")


In [3]:
from google import genai
import os

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

###Call LLM

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    google_api_key=os.getenv("GOOGLE_API_KEY"),
)
llm_response = llm.invoke("Tell me a joke")

print(llm_response.content)

c:\Users\Ronak bansal\Desktop\AI engineering\RAG-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[{'type': 'text', 'text': "Why don't scientists trust atoms?\n\nBecause they make up everything!", 'extras': {'signature': 'EjQKMgERTTIPDoanBuxdkmz9lMNQ4LLscIkIMc7Its77fhidxg4JKeMliYC08ZB9f+mfFBS9'}}]


###Parsing Output

In [5]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
output_parser.invoke(llm_response)

"Why don't scientists trust atoms?\n\nBecause they make up everything!"

###Simple Chain

In [8]:
chain = llm | output_parser
chain.invoke("Tell me a joke")

"Why don't scientists trust atoms?\n\nBecause they make up everything!"

###Structured Output

In [5]:
from typing import List
from pydantic import BaseModel, Field

class MobileReview(BaseModel):
    phone_model: str = Field(description="Name and model of the phone")
    rating: float = Field(description="Overall rating out of 5")
    pros: List[str] = Field(description="List of positive aspects")
    cons: List[str] = Field(description="List of negative aspects")
    summary: str = Field(description="Brief summary of the review")

review_text = """
Finally got the Nothing Phone 4b in my hands and it's a fun little device! The Glyph
lighting on the back still turns heads - never gets old showing it off. Camera's
surprisingly capable for the price, daylight shots are crisp and colors feel natural,
not oversaturated. Performance is snappy for everyday stuff, apps open fast, no lag
scrolling through social media. Battery easily gets me through a full day with juice
to spare.

That said, a few things bug me. Low-light photos are just okay, nothing special,
grainy if you look close. The transparent design looks cool but shows fingerprints
like crazy, wiping it down constantly. Also wish it came with more storage options
at this price point.

Overall, I'd give it a solid 4 out of 5. Great value phone with a unique look, just a
few rough edges keep it from being flawless. If you want something different from the
usual black slab, this one's worth a look!
"""

structured_llm = llm.with_structured_output(MobileReview)
output = structured_llm.invoke(review_text)
output

MobileReview(phone_model='Nothing Phone 4b', rating=4.0, pros=['Unique Glyph lighting design', 'Capable daylight camera', 'Snappy performance', 'Good battery life'], cons=['Mediocre low-light photography', 'Fingerprint magnet', 'Limited storage options'], summary='A stylish and high-performing device that offers great value, though it is held back by average low-light camera performance and limited storage choices.')

In [6]:
output.pros

['Unique Glyph lighting design',
 'Capable daylight camera',
 'Snappy performance',
 'Good battery life']

###Prompt Template

In [8]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("Tell me a dark joke about {topic}")
prompt.invoke({"topic": "messi"})

ChatPromptValue(messages=[HumanMessage(content='Tell me a dark joke about messi', additional_kwargs={}, response_metadata={})])

In [9]:
chain = prompt | llm | output_parser
chain.invoke({"topic": "messi"})

'Why is it so hard for Lionel Messi to play hide and seek?\n\nBecause every time he tries to hide, he realizes he’s spent his whole life being chased by defenders, and now he’s just terrified of being alone.'

###LLM Messages

In [16]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

system_message = SystemMessage(content="You are a helpful assistant that tells jokes.")
human_message = HumanMessage(content="Tell me about kids")
llm.invoke([system_message, human_message])

AIMessage(content=[{'type': 'text', 'text': 'Kids are like tiny, chaotic roommates who don\'t pay rent and have very questionable taste in interior design (mostly involving crayons on the walls).\n\nHere are a few jokes about the joys of raising them:\n\n**The classic observation:**\n"My kid asked me what it’s like to be a parent, so I turned off the TV, threw away his favorite toy, and told him he had to go to bed in five minutes. Then I started crying for no reason."\n\n**The logic:**\n"I asked my son, \'Why are you crying?\' He said, \'Because I don\'t know why I\'m crying.\' Honestly, I’ve never related to a human being more in my entire life."\n\n**The bedtime struggle:**\n"Parenting is just a series of \'Go to sleep\' followed by \'I need water,\' \'I need a snack,\' \'I’m scared of the dark,\' and \'I need to tell you a story about a dinosaur that happened three years ago.\'"\n\n**The "help":**\n"Cleaning your house while your kids are still awake is like trying to brush your te

In [11]:
template = ChatPromptTemplate([
    ("system", "You are a helpful assistant that tells jokes."),
    ("human", "Tell me about {topic}")
])

prompt_value = template.invoke(
    {
        "topic": "programming"
    }
)
prompt_value

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant that tells jokes.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me about programming', additional_kwargs={}, response_metadata={})])

In [12]:
llm.invoke(prompt_value)

AIMessage(content=[{'type': 'text', 'text': 'Programming is a lot like magic, except instead of waving a wand, you spend hours staring at a screen, typing cryptic symbols, and wondering why your "spell" is throwing an error.\n\nHere are a few jokes to capture the essence of the programmer\'s life:\n\n***\n\n**The Classic:**\nA programmer’s spouse says, "While you’re at the store, get some milk."\nThe programmer never returns. They are still at the store, stuck in an infinite loop.\n\n**The Optimist vs. The Pessimist:**\nAn optimist says, "The glass is half full."\nA pessimist says, "The glass is half empty."\nA programmer says, "The glass is twice as large as it needs to be."\n\n**The Debugging Process:**\nProgramming is 10% writing code and 90% understanding why it doesn\'t work.\nIt’s a lot like being a detective in a crime drama where you are also the murderer.\n\n**The Hardware/Software Divide:**\nWhy do programmers prefer dark mode?\nBecause light attracts bugs.\n\n**The Reality C

In [13]:
# !pip install docx2txt pypdf unstructured

In [14]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from typing import List
from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

docx_loader = Docx2txtLoader("docs/GreenGrow Innovations_ Company History.docx")
documents = docx_loader.load()

print(len(documents))

splits = text_splitter.split_documents(documents)

print(f"Split the documents into {len(splits)} chunks.")

C:\Users\Ronak bansal\AppData\Local\Temp\ipykernel_17192\1080345529.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader


1
Split the documents into 2 chunks.


In [26]:
documents[0]

Document(metadata={'source': 'docs/GreenGrow Innovations_ Company History.docx'}, page_content="GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.\n\n\n\nIn its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture. Their first product, the WaterWise Sensor, was launched in 2012 and quickly gained popularity among local farmers. This success allowed the company to expand its research and development efforts.\n\n\n\nBy 2015, GreenGrow had outgrown its garage origins and moved into a proper office and research facility in the outskirts of Portland. This move coincided with the development of their second major product, the SoilHealth Monitor, which used advanced sensors to analyze 

In [ ]:
splits[1]

Document(metadata={'source': '/content/docs/GreenGrow Innovations_ Company History.docx'}, page_content="The company's breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.\n\n\n\nToday, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.\n\n\n\nDespite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research institutions to advance the field of agricultura

In [ ]:
splits[0].metadata

{'source': '/content/docs/GreenGrow Innovations_ Company History.docx'}

In [ ]:
splits[0].page_content

'GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.\n\n\n\nIn its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture. Their first product, the WaterWise Sensor, was launched in 2012 and quickly gained popularity among local farmers. This success allowed the company to expand its research and development efforts.\n\n\n\nBy 2015, GreenGrow had outgrown its garage origins and moved into a proper office and research facility in the outskirts of Portland. This move coincided with the development of their second major product, the SoilHealth Monitor, which used advanced sensors to analyze soil composition and provide real-time recommendations for optimal crop growth.'

In [ ]:
# import nltk
# nltk.download('punkt')

In [28]:
# 1. Function to load documents from a folder

def load_documents(folder_path: str) -> List[Document]:
    documents = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if filename.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
        elif filename.endswith('.docx'):
            loader = Docx2txtLoader(file_path)
        else:
            print(f"Unsupported file type: {filename}")
            continue
        documents.extend(loader.load())
    return documents

# Load documents from a folder
folder_path = "docs"
documents = load_documents(folder_path)

print(f"Loaded {len(documents)} documents from the folder.")
splits = text_splitter.split_documents(documents)
print(f"Split the documents into {len(splits)} chunks.")

Loaded 5 documents from the folder.
Split the documents into 8 chunks.


In [34]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

In [31]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

for model in client.models.list():
    if "embedding" in model.name.lower():
        print(model.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [35]:


# 4. Embedding Documents

document_embeddings = embeddings.embed_documents([split.page_content for split in splits])

print(f"Created embeddings for {len(document_embeddings)} document chunks.")

Created embeddings for 8 document chunks.


In [36]:
document_embeddings[0]

[0.026385719,
 0.029772826,
 0.0025474103,
 -0.005509703,
 -0.000266702,
 -0.0048933187,
 -0.017947743,
 -0.0065298514,
 -0.012043238,
 -0.06291138,
 -0.019682927,
 0.0013195362,
 0.010789265,
 0.0013082752,
 -0.014470896,
 -0.00059713284,
 0.0085576195,
 0.00037206244,
 -0.019328589,
 -0.0023065982,
 -0.0089514665,
 0.0158072,
 0.026582979,
 -0.010850496,
 -0.016521553,
 0.013575645,
 0.023543388,
 0.00072918,
 -0.00061563874,
 0.14146183,
 -0.009443037,
 -0.008036731,
 -0.0046595777,
 0.013359475,
 0.0027190302,
 0.015616161,
 -0.00023053365,
 -0.016077703,
 0.0035576888,
 -0.012902331,
 0.0111098895,
 -0.004526987,
 0.014834163,
 0.0032435295,
 -0.0009950119,
 -0.005280696,
 0.007441837,
 0.032512106,
 -0.010786089,
 -0.004258655,
 0.0031142493,
 0.026121058,
 0.021510258,
 -0.0033493866,
 0.0028968079,
 -0.0209332,
 0.010429088,
 -0.019948877,
 0.006207204,
 0.0016558352,
 0.0052600857,
 0.007487397,
 0.014673199,
 0.00095654285,
 0.023168191,
 0.0030022392,
 0.010622476,
 -0.00359

In [37]:
# !pip install sentence_transformers
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
document_embeddings = embedding_function.embed_documents([split.page_content for split in splits])
document_embeddings[0]

C:\Users\Ronak bansal\AppData\Local\Temp\ipykernel_21320\3719339062.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2641.57it/s]


[0.0293357465416193,
 -0.027797531336545944,
 -0.01846340484917164,
 -0.058924250304698944,
 0.08951887488365173,
 -0.026082225143909454,
 -0.10806205123662949,
 0.03759301081299782,
 -0.005008654668927193,
 -0.041687216609716415,
 -0.016387708485126495,
 -0.02612929232418537,
 -0.0313599668443203,
 0.010636315681040287,
 -0.07282692939043045,
 -0.0030148024670779705,
 -0.01876574382185936,
 -0.01873071677982807,
 0.0665455013513565,
 -0.07033520936965942,
 0.003246106207370758,
 -0.01593754068017006,
 0.06552284955978394,
 0.014457416720688343,
 -0.011876809410750866,
 0.10337897390127182,
 -0.005314045585691929,
 0.001742130727507174,
 0.0036549982614815235,
 -0.03610217198729515,
 -0.012576743960380554,
 0.04590792953968048,
 0.030862504616379738,
 -0.05400508642196655,
 0.02634744718670845,
 0.08197516947984695,
 -0.03140775114297867,
 -0.04810931161046028,
 0.04922591149806976,
 -0.0024446044117212296,
 0.005876422859728336,
 -0.01402465533465147,
 -0.003966060001403093,
 -0.05092

In [38]:
# !pip install langchain_chroma -q

In [39]:
# import shutil

# shutil.rmtree('chroma_db')

###Create and persist Chroma vector store

In [42]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()

embedding_function = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

collection_name = "my_collection"

vectorstore = Chroma.from_documents(
    collection_name=collection_name,
    documents=splits,
    embedding=embedding_function,
    persist_directory="./chroma_db"
)

print("Vector store created and persisted to './chroma_db'")

Vector store created and persisted to './chroma_db'


In [43]:
# 5. Perform similarity search

query = "When was GreenGrow Innovations founded?"
search_results = vectorstore.similarity_search(query, k=2)

print(f"\nTop 2 most relevant chunks for the query: '{query}'\n")
for i, result in enumerate(search_results, 1):
    print(f"Result {i}:")
    print(f"Source: {result.metadata.get('source', 'Unknown')}")
    print(f"Content: {result.page_content}")
    print()


Top 2 most relevant chunks for the query: 'When was GreenGrow Innovations founded?'

Result 1:
Source: docs\GreenGrow Innovations_ Company History.docx
Content: GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.



In its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture. Their first product, the WaterWise Sensor, was launched in 2012 and quickly gained popularity among local farmers. This success allowed the company to expand its research and development efforts.



By 2015, GreenGrow had outgrown its garage origins and moved into a proper office and research facility in the outskirts of Portland. This move coincided with the development of their second major product, the S

In [44]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

retriever.invoke("When was GreenGrow Innovations founded?")

[Document(id='9c7e7c3b-99b5-40b3-aa66-428629e9f6a2', metadata={'source': 'docs\\GreenGrow Innovations_ Company History.docx'}, page_content='GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.\n\n\n\nIn its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture. Their first product, the WaterWise Sensor, was launched in 2012 and quickly gained popularity among local farmers. This success allowed the company to expand its research and development efforts.\n\n\n\nBy 2015, GreenGrow had outgrown its garage origins and moved into a proper office and research facility in the outskirts of Portland. This move coincided with the development of their second major product, the SoilHealth Mon

In [45]:
from langchain_core.prompts import ChatPromptTemplate
template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer: """
prompt = ChatPromptTemplate.from_template(template)

In [47]:
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()} | prompt
)
rag_chain.invoke("When was GreenGrow Innovations founded?")

ChatPromptValue(messages=[HumanMessage(content='Answer the question based only on the following context:\n[Document(id=\'9c7e7c3b-99b5-40b3-aa66-428629e9f6a2\', metadata={\'source\': \'docs\\\\GreenGrow Innovations_ Company History.docx\'}, page_content=\'GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.\\n\\n\\n\\nIn its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture. Their first product, the WaterWise Sensor, was launched in 2012 and quickly gained popularity among local farmers. This success allowed the company to expand its research and development efforts.\\n\\n\\n\\nBy 2015, GreenGrow had outgrown its garage origins and moved into a proper office and research facili

In [48]:
def docs2str(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [49]:
rag_chain = (
    {"context": retriever | docs2str, "question": RunnablePassthrough()} | prompt
)
rag_chain.invoke("When was GreenGrow Innovations founded?")

ChatPromptValue(messages=[HumanMessage(content="Answer the question based only on the following context:\nGreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.\n\n\n\nIn its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture. Their first product, the WaterWise Sensor, was launched in 2012 and quickly gained popularity among local farmers. This success allowed the company to expand its research and development efforts.\n\n\n\nBy 2015, GreenGrow had outgrown its garage origins and moved into a proper office and research facility in the outskirts of Portland. This move coincided with the development of their second major product, the SoilHealth Monitor, which used advanced sensors t

In [50]:
rag_chain = (
    {"context": retriever | docs2str, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
question = "When was GreenGrow Innovations founded?"
response = rag_chain.invoke(question)
print(response)

GreenGrow Innovations was founded in 2010.


###Conversational RAG

####Handling Follow Up Questions

In [51]:
# Example conversation
from langchain_core.messages import HumanMessage, AIMessage
chat_history = []
chat_history.extend([
    HumanMessage(content=question),
    AIMessage(content=response)
])

In [52]:
chat_history

[HumanMessage(content='When was GreenGrow Innovations founded?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='GreenGrow Innovations was founded in 2010.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [53]:
from langchain_core.prompts import MessagesPlaceholder
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# history_aware_retriever = create_history_aware_retriever(
#     llm, retriever, contextualize_q_prompt
# )
contextualize_chain = contextualize_q_prompt | llm | StrOutputParser()
contextualize_chain.invoke({"input": "Where it is headquartered?", "chat_history": chat_history})

'Where is GreenGrow Innovations headquartered?'

In [56]:

from langchain_classic.chains import create_history_aware_retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever.invoke({"input": "Where it is headquartered?", "chat_history": chat_history})

[Document(id='9d39b764-2f59-405d-9675-38586e7fcb08', metadata={'source': 'docs\\GreenGrow Innovations_ Company History.docx'}, page_content="The company's breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.\n\n\n\nToday, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.\n\n\n\nDespite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research institutions

In [57]:
retriever.invoke("Where it is headquartered?")

[Document(id='9ecdcf6f-ddcb-4c65-807b-d4633d6bf8af', metadata={'source': 'docs\\Company_ TechWave Innovations.docx'}, page_content='Company: TechWave Innovations\n\nHeadquarters: TechWave Innovations is headquartered in San Francisco, California, USA. As a leader in cutting-edge AI and machine learning solutions, the company thrives in the heart of Silicon Valley, benefiting from its proximity to tech giants and a dynamic startup ecosystem. With its headquarters in this global technology hub, TechWave Innovations has access to top talent and a vast network of innovation-driven enterprises.'),
 Document(id='09bcf0a5-08f6-4102-b8ee-62abd08ef460', metadata={'source': 'docs\\Company_ QuantumNext Systems.docx'}, page_content='Company: QuantumNext Systems\n\nHeadquarters: QuantumNext Systems is headquartered in Bangalore, Karnataka, India. The company, specializing in quantum computing and advanced data processing, is situated in the bustling tech metropolis of Bangalore, often referred to a

In [58]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the following context to answer the user's question."),
    #  ("system", "Tell me joke on Programming"),
    ("system", "Context: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [59]:
rag_chain.invoke({"input": "Where it is headquartered?", "chat_history":chat_history})

{'input': 'Where it is headquartered?',
 'chat_history': [HumanMessage(content='When was GreenGrow Innovations founded?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='GreenGrow Innovations was founded in 2010.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'context': [Document(id='9d39b764-2f59-405d-9675-38586e7fcb08', metadata={'source': 'docs\\GreenGrow Innovations_ Company History.docx'}, page_content="The company's breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.\n\n\n\nToday, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural t

###Building Multi User Chatbot

In [60]:
import sqlite3
from datetime import datetime

DB_NAME = "rag_app.db"

def get_db_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    return conn

def create_application_logs():
    conn = get_db_connection()
    conn.execute('''CREATE TABLE IF NOT EXISTS application_logs
                    (id INTEGER PRIMARY KEY AUTOINCREMENT,
                     session_id TEXT,
                     user_query TEXT,
                     gpt_response TEXT,
                     model TEXT,
                     created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    conn.close()

def insert_application_logs(session_id, user_query, gpt_response, model):
    conn = get_db_connection()
    conn.execute('INSERT INTO application_logs (session_id, user_query, gpt_response, model) VALUES (?, ?, ?, ?)',
                 (session_id, user_query, gpt_response, model))
    conn.commit()
    conn.close()

def get_chat_history(session_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT user_query, gpt_response FROM application_logs WHERE session_id = ? ORDER BY created_at', (session_id,))
    messages = []
    for row in cursor.fetchall():
        messages.extend([
            {"role": "human", "content": row['user_query']},
            {"role": "ai", "content": row['gpt_response']}
        ])
    conn.close()
    return messages

# Initialize the database
create_application_logs()

In [61]:
import uuid
session_id = str(uuid.uuid4())
chat_history = get_chat_history(session_id)
print(chat_history)
question1 = "When was GreenGrow Innovations founded?"
answer1 = rag_chain.invoke({"input": question1, "chat_history":chat_history})['answer']
insert_application_logs(session_id, question1, answer1, "gpt-4-o-mini")
print(f"Human: {question1}")
print(f"AI: {answer1}\n")

[]
Human: When was GreenGrow Innovations founded?
AI: GreenGrow Innovations was founded in 2010.



In [62]:
question2 = "Where it is headquartered?"
chat_history = get_chat_history(session_id)
print(chat_history)
answer2 = rag_chain.invoke({"input": question2, "chat_history":chat_history})['answer']
insert_application_logs(session_id, question2, answer2, "gpt-3.5-turbo")
print(f"Human: {question2}")
print(f"AI: {answer2}\n")

[{'role': 'human', 'content': 'When was GreenGrow Innovations founded?'}, {'role': 'ai', 'content': 'GreenGrow Innovations was founded in 2010.'}]
Human: Where it is headquartered?
AI: The provided text does not explicitly state where GreenGrow Innovations is currently headquartered. It mentions that the company started in a garage in **Portland, Oregon**, and that it has since expanded to include offices in **California and Iowa**, but it does not specify which location serves as the headquarters.



New User

In [63]:
session_id = str(uuid.uuid4())
question = "What is GreenGrow"
chat_history = get_chat_history(session_id)
print(chat_history)
answer = rag_chain.invoke({"input": question, "chat_history":chat_history})['answer']
insert_application_logs(session_id, question, answer, "gpt-3.5-turbo")
print(f"Human: {question}")
print(f"AI: {answer}\n")

[]
Human: What is GreenGrow
AI: GreenGrow Innovations is a company focused on developing sustainable agricultural technologies. 

Key facts about the company include:

*   **Breakthrough:** The company rose to national prominence in 2018 with the introduction of the **EcoHarvest System**, an integrated solution that combines smart irrigation, soil monitoring, and automated harvesting.
*   **Operations:** They employ over 200 people and maintain offices in California and Iowa.
*   **Current Focus:** They are currently working on projects involving vertical farming, drought-resistant crop development, and AI-powered farm management systems.
*   **Mission:** The company is committed to promoting sustainable farming practices and frequently collaborates with universities and research institutions to advance agricultural technology.

